## 1. Setup <a id='setup'></a>

In [ ]:
import json
import random
from pathlib import Path

import numpy as np
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
import rasterio.plot
from rasterio.mask import mask
import math
import seaborn as sns
from PIL import Image
from tqdm import tqdm
from shapely.geometry import Polygon
from typing import Dict, List, Tuple, Any


plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')

# Paths to the three dataset splits
DATA_ROOT = Path('/home/leafline/leafline/Data/Kiel/TrainingAreas')
SUMMER_20 = DATA_ROOT / 'DOP20'
SPRING_20 = DATA_ROOT / 'DOP20-spring'
SPRING_75 = DATA_ROOT / 'DOP7-5'
NDOM_20 = DATA_ROOT / 'nDOM20'
NDOM_75 = DATA_ROOT / 'nDOM7-5'

SPLITS = {'summer 20': SUMMER_20, 'spring 20': SPRING_20, 'spring 7.5': SPRING_75, 'height 20': NDOM_20, 'height 7.5': NDOM_75}

# ---------------------------------------------------------------------
# Root directory of the dataset
# ---------------------------------------------------------------------
data_root = Path("/home/leafline/leafline/Data/Kiel/TrainingAreas")

# Image folders (.tif)
image_folders = {
    "RGBI_20cm_summer": data_root / "DOP20",
    "RGBI_20cm_spring": data_root / "DOP20-spring",
    "RGBI_7_5cm": data_root / "DOP7-5",
    "Height_20cm": data_root / "nDOM20",
    "Height_7_5cm": data_root / "nDOM7-5",
}

# Label folders (.shp)
label_folders = {
    "train": data_root / "train",
    "valid": data_root / "valid",
    "test": data_root / "test",
}

for name, path in image_folders.items():
    print(f"{name}: {path.exists()} - {path}")

for name, path in label_folders.items():
    print(f"{name}: {path.exists()} - {path}")


## 2. Dataset Overview <a id='dataset-overview'></a>

In [ ]:
# ---------------------------------------------------------------------
# Read files
# ---------------------------------------------------------------------
dataset = {}

for folder_name, folder in image_folders.items():
    dataset[folder_name] = sorted(folder.glob("*.tif"))

# Read shapefiles
labels = {}
for split, folder in label_folders.items():
    labels[split] = sorted(folder.glob("*.shp"))

# ---------------------------------------------------------------------
# Print folder contents
# ---------------------------------------------------------------------
print("IMAGE FOLDERS")
for folder, files in dataset.items():
    print(f"\n{folder} ({len(files)} files)")
    for f in files:
        print("  ", f.name)

print("\nLABEL FOLDERS")
for split, files in labels.items():
    print(f"\n{split} ({len(files)} shapefiles)")
    for f in files:
        print("  ", f.name)

In [ ]:
# ---------------------------------------------------------------------
# Get image statistics and summarize
# ---------------------------------------------------------------------
results = []

# results by shapefile
for split, folder in label_folders.items():
    for shp in folder.glob("*.shp"):
        gdf = gpd.read_file(shp)
        n_features = len(gdf)
        total_area = gdf.area.sum()  # m²

        results.append({
            "split": split,
            "shapefile": shp.stem,
            "features": n_features,
            "total_area_m2": gdf.area.sum(),
            "features_per_m2": n_features / total_area if total_area > 0 else None,
   
        })
summary = pd.DataFrame(results)


# overall statistics
image_summary = {
    "largest_image": summary.loc[summary["total_area_m2"].idxmax(), "shapefile"],
    "largest_image_area_m2": summary["total_area_m2"].max(),

    "smallest_image": summary.loc[summary["total_area_m2"].idxmin(), "shapefile"],
    "smallest_image_area_m2": summary["total_area_m2"].min(),

    "max_features_image": summary.loc[summary["features"].idxmax(), "shapefile"],
    "max_number_of_features": summary["features"].max(),
    
    "min_features_image": summary.loc[summary["features"].idxmin(), "shapefile"],
    "min_number_of_features": summary["features"].min(),

    "mean_image_area_m2": summary["total_area_m2"].mean(),
    "mean_number_of_features": summary["features"].mean(),
}
image_summary_df = pd.DataFrame([image_summary])


# statistics summarized by split
summary_by_split = (
    summary.groupby("split")
    .agg(
        shapefiles=("shapefile", "count"),
        features=("features", "sum"),
        total_area_m2=("total_area_m2", "sum"),
    )
    .assign(
        perc_of_total_area=lambda df: df["total_area_m2"] / df["total_area_m2"].sum() * 100,
        features_per_m2=lambda df: df["features"] / df["total_area_m2"],
        perc_of_total_features=lambda df: df["features"] / df["features"].sum() * 100,
    )
    .reset_index()
)

#print statistics summaries
pd.set_option("display.max_columns", None)
pd.options.display.float_format = "{:.2f}".format

print("statistics: \n",image_summary_df.T.to_string(header=False))
print("\nSummary by split:\n", summary_by_split)


# ---------------------------------------------------------------------
# Bar diagram: area and feature count per image
# ---------------------------------------------------------------------

fig, ax1 = plt.subplots(figsize=(12, 6))

x = np.arange(len(summary["shapefile"]))
width = 0.35


# Primary axis: Area
bars1 = ax1.bar(
    x - width/2,
    summary["total_area_m2"],
    width,
    label="Area (m²)",
    color="steelblue",
    zorder=2
)

ax1.set_ylabel("Area (m²)", color="steelblue")
ax1.tick_params(axis="y", labelcolor="steelblue")


# Secondary axis: Feature count
ax2 = ax1.twinx()

bars2 = ax2.bar(
    x + width/2,
    summary["features"],
    width,
    label="Number of features",
    color="darkorange",
    zorder=3
)

ax2.set_ylabel("Number of features", color="darkorange")
ax2.tick_params(axis="y", labelcolor="darkorange")


# Secondary axis grid lines

# Make secondary axis visible without covering the bars
ax2.set_zorder(ax1.get_zorder() + 1)
ax2.patch.set_visible(False)

ax2.grid(
    axis="y",
    linestyle=(0, (5, 5)),   # dashed/striped line style
    linewidth=0.8,
    color="darkorange",
    alpha=0.5,
    zorder=1
)

# X-axis labels
ax1.set_xticks(x)
ax1.set_xticklabels(
    summary["shapefile"],
    rotation=45,
    ha="right"
)

ax1.set_title("Image statistics: area and number of features")

#combined legend
handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(
    handles1 + handles2,
    labels1 + labels2,
    loc="upper right"
)


plt.tight_layout()
plt.show()

**Results:** 
The dataset is split into multiple folders. There are 5 image folders (.tif format), three containing the different RGBI images (different resolution and seasons), and two containing the corresponding height data. 
Furthermore there are 3 label folders containing shapefiles. The files have the same naming scheme as the tif files, but are split into training (4 files), validation (3 files) and testing (2 files).
This amounts to 9 samples, each available as a high-resolution spring image, low-resolution spring image, and low-resolution summer image.

The image size varies considerably with the smallest image covering an area of ~1.313 m² and the largest image covering ~51.801 m². The number of trees per images also varies greatly. While predicably less in smaller images and more in larger ones, HoernSued, HoernNord, and Veloroute are notable for having combarably much more features per area than the other images.
The varying image size means that each sample must be split into patches to be processed by the model.

In [ ]:
# ---------------------------------------------------------------------
# Image overview prep
# ---------------------------------------------------------------------

# normalize names 
def get_sample_name(filename):
    """
    Remove modality-specific suffixes from filenames.
    Adjust the suffix list if your dataset contains more variants.
    """

    name = filename.stem

    suffixes = [
        "_nDOM",
        "_spring",
        "_summer",
        "_DOP20",
        "_DOP7-5",
        "_RGBI",
        "_GroundTruth",
    ]

    for suffix in suffixes:
        if name.endswith(suffix):
            name = name[:-len(suffix)]

    return name


# find images
sample_name = "BotGarten"

matches = {} # to store all files with sample name 

for modality, folder in image_folders.items():
    for tif_file in folder.glob("*.tif"):
        sample = get_sample_name(tif_file)
        if sample_name in sample:
            matches.setdefault(sample, {})[modality] = tif_file

for split, folder in label_folders.items():
    for shp_file in folder.glob("*.shp"):
        sample = get_sample_name(shp_file)
        if sample_name in sample:
            matches.setdefault(sample, {})[split] = shp_file

In [ ]:
# ---------------------------------------------------------------------
# Image overview display
# ---------------------------------------------------------------------

image_keys = [
    "RGBI_20cm_summer",
    "RGBI_20cm_spring",
    "RGBI_7_5cm",
    "Height_20cm",
    "Height_7_5cm",
]

fig, axes = plt.subplots(1, len(image_keys) + 1, figsize=(25, 5))

# Display all TIFs
for ax, key in zip(axes[:-1], image_keys):
    path = matches[sample_name].get(key)

    if path is None:
        ax.set_title(f"{key}\nmissing")
        ax.axis("off")
        continue

    with rasterio.open(path) as src:
        img = src.read()

    if img.shape[0] >= 3:
        # RGB display (ignore NIR band if present)
        display_img = img[:3].transpose(1, 2, 0).astype(float)

        # Contrast stretch
        display_img -= display_img.min()
        display_img /= display_img.max()

        ax.imshow(display_img)

    else:
        # Height data
        ax.imshow(img[0], cmap="terrain")

    ax.set_title(key)
    ax.axis("off")


# Display summer 20cm image + shapefile overlay
ax = axes[-1]

summer_path = matches[sample_name]["RGBI_20cm_summer"]
#shapefile path either in train, valid or test
for split in label_folders:
    if split in matches.get(sample_name, {}):
        shp_path = matches[sample_name].get(split)
        break

#raster
with rasterio.open(summer_path) as src:
    rgb = src.read([1, 2, 3]).transpose(1, 2, 0)
    extent = rasterio.plot.plotting_extent(src)

    rgb = rgb.astype(float)
    rgb -= rgb.min()
    rgb /= rgb.max()

    ax.imshow(rgb, extent=extent)

#overlay shp
if shp_path:
    labels = gpd.read_file(shp_path)

    # Ensure same coordinate system as raster
    with rasterio.open(summer_path) as src:
        if labels.crs != src.crs:
            labels = labels.to_crs(src.crs)

    labels.plot(
        ax=ax,
        facecolor="none",
        edgecolor="red",
        linewidth=1.5
    )

ax.set_title("RGBI_20cm_summer + labels")
ax.axis("off")

plt.tight_layout()
plt.show()

**Results:** The image overview clearly shows the difference between the summer and spring images. The trees in the summer image are much clearer seperated from each other and the ground. Notable are also the shadows, indicating different times of recording/ the influence of the season on the solar angle. The spring images show larger shadows.

In the height channel both large trees and buildings are clearly differentiated from the background. Smaller trees (e.g. on the right side and at the bottom of the image) are less clear. This might make it more difficult to detect small trees compared to big ones.

## 3. Handling Missing Values <a id='handling-missing-values'></a>

For image datasets, missing values mean: missing files per sample, or invalid pixel values (NaN) in float channels like NDVI and CHM.

[Identify any missing values in the dataset, and describe your approach to handle them if there are any. If there are no missing values simply indicate that there are none.]

In [ ]:
# ---------------------------------------------------------------------
# Normalize filenames and check matching samples
# ---------------------------------------------------------------------

def get_sample_name(filename):
    """
    Remove modality-specific suffixes from filenames.
    Adjust the suffix list if your dataset contains more variants.
    """

    name = filename.stem

    suffixes = [
        "_nDOM",
        "_spring",
        "_summer",
        "_DOP20",
        "_DOP7-5",
        "_RGBI",
        "_GroundTruth",
    ]

    for suffix in suffixes:
        if name.endswith(suffix):
            name = name[:-len(suffix)]

    return name


def check_matching_samples(image_folders):

    folder_samples = {}

    for folder_name, folder in image_folders.items():

        samples = {}

        for tif in folder.glob("*.tif"):
            sample_name = get_sample_name(tif)
            samples[sample_name] = tif.name

        folder_samples[folder_name] = samples


    # All unique samples
    all_samples = set.union(*[
        set(samples.keys())
        for samples in folder_samples.values()
    ])

    print("Total unique samples:", len(all_samples))


    print("\nMissing samples per folder:")

    missing_table = []

    for folder_name, samples in folder_samples.items():

        missing = all_samples - set(samples.keys())

        print(f"\n{folder_name}: {len(missing)} missing")

        for sample in sorted(missing):
            print("  ", sample)

        for sample in missing:
            missing_table.append({
                "folder": folder_name,
                "missing_sample": sample
            })


    return folder_samples, pd.DataFrame(missing_table)


folder_samples, missing_samples = check_matching_samples(image_folders)
display(missing_samples)

In [ ]:
# ---------------------------------------------------------------------
# Check shapefile split consistency
# ---------------------------------------------------------------------

def check_shapefile_splits(label_folders):
    """
    Checks that every sample exists in exactly one of train/valid/test.
    Uses filename normalization to remove suffixes.
    """

    split_samples = {}

    # Collect shapefile samples per split
    for split, folder in label_folders.items():

        samples = {}

        for shp in folder.glob("*.shp"):
            sample_name = get_sample_name(shp)
            samples[sample_name] = shp.name
            
        split_samples[split] = samples


    # All samples across all splits
    all_samples = set.union(*[
        set(samples.keys())
        for samples in split_samples.values()
    ])

    print("Total unique shapefile samples:", len(all_samples))


    # Check samples appearing in multiple splits
    duplicates = []

    for sample in sorted(all_samples):
        locations = [
            split
            for split, samples in split_samples.items()
            if sample in samples
        ]
        
        if len(locations) > 1:
            duplicates.append({
                "sample": sample,
                "splits": locations
            })

    duplicates_df = pd.DataFrame(duplicates)


    # Check samples missing from all splits (only useful if compared to images)
    split_counts = pd.DataFrame({
        "sample": list(all_samples)
    })

    split_counts["number_of_splits"] = split_counts["sample"].apply(
        lambda x: sum(
            x in samples
            for samples in split_samples.values()
        )
    )


    print("\nSamples occurring in multiple splits:")
    print(f"  {len(duplicates_df)}")

    if len(duplicates_df) > 0:
        display(duplicates_df)


    print("\nSamples with invalid split count:")
    invalid = split_counts[split_counts["number_of_splits"] != 1]

    display(invalid)


    return split_samples, duplicates_df, invalid


split_samples, duplicate_labels, invalid_labels = check_shapefile_splits(label_folders)

In [ ]:
# ---------------------------------------------------------------------
# Check raster images for missing pixel values
# ---------------------------------------------------------------------

def check_raster_missing_values(image_folders):
    """
    Checks all tif images for missing/invalid pixel values.
    """

    problems = []

    for folder_name, folder in image_folders.items():

        print(f"\nChecking {folder_name}...")

        for tif in tqdm(sorted(folder.glob("*.tif"))):

            with rasterio.open(tif) as src:

                data = src.read()

                nodata = src.nodata

                # Check invalid pixels
                nan_pixels = np.isnan(data).sum()
                inf_pixels = np.isinf(data).sum()

                if nodata is not None:
                    nodata_pixels = np.sum(data == nodata)
                else:
                    nodata_pixels = 0

                total_pixels = data.size

                invalid_pixels = (
                    nan_pixels +
                    inf_pixels +
                    nodata_pixels
                )

                invalid_percentage = (
                    invalid_pixels / total_pixels * 100
                )

                if invalid_pixels > 0:
                    problems.append({
                        "folder": folder_name,
                        "image": tif.name,
                        "bands": src.count,
                        "width": src.width,
                        "height": src.height,
                        "nodata_value": nodata,
                        "nan_pixels": nan_pixels,
                        "inf_pixels": inf_pixels,
                        "nodata_pixels": nodata_pixels,
                        "invalid_percentage": invalid_percentage
                    })


    return pd.DataFrame(problems)


missing_values_df = check_raster_missing_values(image_folders)

print("\nImages containing missing values:")
display(missing_values_df)

**Results:** All expected files are present. Each file is mapped to a shapefile with the same name containing the ground truth data. The shapefiles are split into train, val, and test with no overlap, missing files or unexpected extra files. 
No images contain missing values. 

## 4. Feature Distributions <a id='feature-distributions'></a>

In [ ]:
# ---------------------------------------------------------------------
# Feature distribution in shapefiles
# ---------------------------------------------------------------------

def extract_shape_features(label_folders):

    records = []

    for split, folder in label_folders.items():

        for shp in folder.glob("*.shp"):

            gdf = gpd.read_file(shp)

            for idx, row in gdf.iterrows():

                geom = row.geometry

                # Skip invalid or empty geometries
                if geom is None or geom.is_empty:
                    continue

                area = geom.area
                perimeter = geom.length

                bounds = geom.bounds
                width = bounds[2] - bounds[0]
                height = bounds[3] - bounds[1]

                compactness = (
                    4 * np.pi * area / perimeter**2
                    if perimeter > 0 else 0
                )

                # Count vertices for Polygon and MultiPolygon
                if geom.geom_type == "Polygon":
                    vertices = len(geom.exterior.coords)

                elif geom.geom_type == "MultiPolygon":
                    vertices = sum(
                        len(poly.exterior.coords)
                        for poly in geom.geoms
                    )

                else:
                    vertices = np.nan


                records.append({

                    "split": split,
                    "sample": shp.stem,

                    "geometry_type": geom.geom_type,

                    "area_m2": area,
                    "perimeter_m": perimeter,

                    # "bbox_width": width,
                    # "bbox_height": height,

                    # "compactness": compactness,

                    # "vertices": vertices
                })


    return pd.DataFrame(records)


shape_features = extract_shape_features(label_folders)

shape_features.head()

In [ ]:
# statistics summary
print(
    f"Total number of tree crowns: {len(shape_features)} \n"
    f"Mean crown area: {shape_features['area_m2'].mean():.2f} m²\n"
    f"Median crown area: {shape_features['area_m2'].median():.2f} m²\n"
    f"Maximum crown area: {shape_features['area_m2'].max():.2f} m²\n"
    f"Skewness: {shape_features['area_m2'].skew():.2f}"
)

In [ ]:
# Distribution plot of tree crown area

numeric_features = [
    "area_m2",
]

for feature in numeric_features:

    plt.figure(figsize=(8,4))

    sns.histplot(
        shape_features[feature],
        bins=50,
        kde=True
    )

    plt.title(
        f"Crown Area Distribution"
    )

    plt.xlabel("Crown Area [m²]")
    plt.ylabel("Count")

    plt.show()

In [ ]:
# tree crown area outliers
for feature in numeric_features:

    plt.figure(figsize=(8,2))

    sns.boxplot(
        x=shape_features[feature]
    )

    plt.title(f"Outliers: Crown Area")
    
    plt.xlabel("Crown Area [m²]")

    plt.show()

**Results:** There are a total of 3178 tree crowns in the dataset. The distribution of the crown area shows a right-skewed distribution, with most tree crowns being small and a small number being larger, which is expected. The median crown area is 29.68 m², which (assuming a perfectly round area) corresponds to a crown radius of 3.07 m. The mean crown area of 45.92 m² corresponds to a radius of 3.82 m. These values are realistic.   
The largest tree crown has an area of 744.79 m², which would mean a crown radius of 15.40 m. While not impossible this seems very large and might also stem from a tree group being wrongly marked as a single tree. 
The outliers show, that only three crowns in the dataset are above an area of 400 m² (radius 11m ). Since most tree crowns have an area under 100 m² and only very few outliers one above 400 m², the dataset can be assumed to be realistic and requires no changes.

In [ ]:
# ---------------------------------------------------------------------
# Feature distribution in TIFs (pixel count stats) - Setup
# ---------------------------------------------------------------------

CHANNEL_NAMES = ["Red", "Green", "Blue", "Infrared"]
CHANNEL_COLORS = ["red", "green", "blue", "purple"]
HEIGHT_COLOR = "#00aaff"  # bright cyan-blue

RGBI_BINS = 256 # 0-255
HEIGHT_BINS = np.arange(-40, 50, 1) #1m bins

HistogramResult = Tuple[np.ndarray, np.ndarray, np.ndarray]
DatasetGroup = List[Tuple[str, HistogramResult]]


def get_resolution(folder: Path) -> Tuple[float, float]:
    """Return raster pixel resolution as a normalized tuple."""
    tif_path = sorted(folder.glob("*.tif"))[0]

    with rasterio.open(tif_path) as src:
        resolution = (
            abs(src.transform.a),
            abs(src.transform.e),
        )
        
    return tuple(round(value, 6) for value in resolution)


def calculate_ymax(datasets: DatasetGroup, band_index: int | None = None) -> float:
    """Calculate a shared y-axis limit with a small margin."""
    if band_index is None:
        maximum = max(
            histograms.max()
            for _, (histograms, _, _) in datasets
        )
    else:
        maximum = max(
            histograms[band_index].max()
            for _, (histograms, _, _) in datasets
        )

    return maximum * 1.05


def compute_image_histograms(folder: Path, bins):
    """
    Compute accumulated histograms for all TIFFs in a folder.

    Returns
    -------
    histograms : ndarray (bands, bins)
    centers    : ndarray (bands, bins)
    edges      : ndarray (bands, bins + 1)
    """
    tif_files = sorted(folder.glob("*.tif"))

    if not tif_files:
        raise ValueError(f"No TIFF files found in {folder}")

    accumulated = None
    centers = None
    edges = None

    for tif in tif_files:
        with rasterio.open(tif) as src:

            if accumulated is None:
                accumulated = []

                for band in range(src.count):
                    accumulated.append(np.zeros(len(np.histogram([], bins=bins)[0]), dtype=np.int64))

            for band in range(src.count):
                data = src.read(band + 1)

                # remove nodata values
                if src.nodata is not None:
                    data = data[data != src.nodata]

                data = data[np.isfinite(data)]

                hist, edge = np.histogram(data, bins=bins)

                accumulated[band] += hist

                if centers is None:
                    centers = []
                    edges = []

                if len(centers) <= band:
                    edges.append(edge)
                    centers.append((edge[:-1] + edge[1:]) / 2)

    return (
        np.asarray(accumulated),
        np.asarray(centers),
        np.asarray(edges),
    )


def configure_axis(
    ax: plt.Axes,
    title: str,
    xlabel: str,
    ymax: float,
    ylabel: str | None = None,
) -> None:
    """Apply common histogram axis settings."""
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylim(0, ymax)

    if ylabel:
        ax.set_ylabel(ylabel)


def plot_rgbi_histograms(
    datasets: DatasetGroup,
    ymax: float,
) -> None:
    """Plot RGBI histograms for datasets with matching resolution."""
    for name, (histograms, centers, edges) in datasets:
        fig, axes = plt.subplots(
            1,
            histograms.shape[0],
            figsize=(18, 4),
            sharey=True,
        )

        for index, ax in enumerate(np.atleast_1d(axes)):
            ax.bar(
                centers[index],
                histograms[index],
                width=np.diff(edges[index]),
                color=CHANNEL_COLORS[index],
            )

            configure_axis(
                ax,
                title=CHANNEL_NAMES[index],
                xlabel="Pixel value",
                ymax=ymax,
                ylabel="Pixel count" if index == 0 else None,
            )

        fig.suptitle(name, fontsize=14)
        plt.tight_layout()
        plt.show()


def plot_height_histograms(
    datasets: DatasetGroup,
    ymax: float,
) -> None:
    """Plot height histograms for datasets with matching resolution."""
    fig, axes = plt.subplots(
        1,
        len(datasets),
        figsize=(18, 4),
        sharey=True,
    )

    for ax, (name, (histograms, centers, edges)) in zip(
        np.atleast_1d(axes),
        datasets,
    ):
        ax.bar(
            centers[0],
            histograms[0],
            width=np.diff(edges[0]),
            color=HEIGHT_COLOR,
        )

        configure_axis(
            ax,
            title=name,
            xlabel="Height value",
            ylabel="Pixel count",
            ymax=ymax,
        )

    plt.tight_layout()
    plt.show()


def process_datasets(
    image_folders: Dict[str, Path],
) -> Tuple[Dict[Tuple[float, float], DatasetGroup],
           Dict[Tuple[float, float], DatasetGroup]]:
    """Compute histograms and group datasets by resolution."""
    rgbi_results: Dict[Tuple[float, float], DatasetGroup] = {}
    height_results: Dict[Tuple[float, float], DatasetGroup] = {}

    for name, folder in image_folders.items():
        print(f"Processing {name}")

        bins = (
            RGBI_BINS
            if "RGBI" in name.upper()
            else HEIGHT_BINS
        )

        result = compute_image_histograms(
            folder,
            bins=bins,
        )

        resolution = get_resolution(folder)

        target = (
            rgbi_results
            if result[0].shape[0] > 1
            else height_results
        )

        target.setdefault(resolution, []).append(
            (name, result)
        )

    return rgbi_results, height_results

In [ ]:
# -------------------------------------------------------------
# Feature distribution in TIFs (pixel count stats) - Processing
# -------------------------------------------------------------

rgbi_results, height_results = process_datasets(image_folders)


for resolution, datasets in rgbi_results.items():
    ymax = calculate_ymax(datasets)

    print(
        f"RGBI resolution {resolution}: y-axis max = {ymax}"
    )

    plot_rgbi_histograms(
        datasets,
        ymax,
    )


for resolution, datasets in height_results.items():
    ymax = calculate_ymax(
        datasets,
        band_index=0,
    )

    print(
        f"Height resolution {resolution}: y-axis max = {ymax}"
    )

    plot_height_histograms(
        datasets,
        ymax,
    )

**Results:** The RGBI distibution between the two spring resolutions is the same, as it should be. The resampling did not distort the data. 
The difference between the summer and spring images is clear. The summer images have a spread out distribution in the RGB channels with two peaks each. The spring images in comarison are left-skewed in the RGB channels. This is largely due to the seasonal difference, but might also be influenced by the time of recording. The spring images show larger shadows, which also tend to move an image towards darker, blue/greener values.

The height distribution is centered near zero (flat ground, fields, roads), with a slight right-skew and a sparse tail extending to ~25 m (tall trees and buildings). It also shows some negative values, which indicate the water areas of the Kieler Förde, which appears in some images.

## 5. Possible Biases <a id='possible-biases'></a>

**Regional Bias:** Ground-Truth data is only collected within the city of Kiel. This will likely limit the performance in rural contexts, other climate zones with differnt trees, and mountainous regions. 

**Seasonal Bias:** Most models are trained on summer (leaf-on) imagery, since it allows for much better tree identification and segmentation. Since aerial images collected in Kiel city are generally collected in spring (leaf-off), a model will require finetuning to accuratly capture trees in sping images. The dataset contains these images.

**Resolution Bias:** Most models are trained on resolutions around 20cm, since public data is not available at higher resolution. This dataset contains high-resolution (7.5cm) images from spring 2025. 

This dataset contains images from multiple seasons and with multiple resolutions and is therefore well suited to analyze the impact of seasonal and resolution bias on model output.

## 6. Correlations <a id='correlations'></a>

We compute per-image mean values for each channel and correlate them with tree count to understand which features are most informative for tree detection.

In [ ]:
# --------------------------------------------------------------
# Setup functions
# --------------------------------------------------------------

def build_lookup(folder):

    lookup = {}
    for tif in folder.glob("*.tif"):
        lookup[get_sample_name(tif)] = tif

    return lookup

def extract_tree_features(
        rgb_folder,
        height_folder,
        label_folders):

    records = []

    rgb_lookup = build_lookup(rgb_folder)
    height_lookup = build_lookup(height_folder)
    

    for split, folder in label_folders.items():

        print("\nProcessing:", split)
        for shp_path in tqdm(
            sorted(folder.glob("*.shp"))
        ):
            sample = get_sample_name(shp_path)
            rgb_path = rgb_lookup.get(sample)
            height_path = height_lookup.get(sample)

            if rgb_path is None:
                print("Missing RGB:", sample)
                continue
                
            if height_path is None:
                print("Missing height:", sample)
                continue

            crowns = gpd.read_file(shp_path)

            with rasterio.open(rgb_path) as rgb_src, \
                 rasterio.open(height_path) as h_src:


                # CRS correction
                if crowns.crs != rgb_src.crs:
                    crowns = crowns.to_crs(
                        rgb_src.crs
                    )

                for tree_id, crown in crowns.iterrows():
                    geom = [
                        crown.geometry
                    ]

                    try:
                        rgb, _ = mask(
                            rgb_src,
                            geom,
                            crop=True
                        )

                        height, _ = mask(
                            h_src,
                            geom,
                            crop=True
                        )

                        # RGBI mask
                        rgb_valid = np.all(
                            np.isfinite(rgb),
                            axis=0
                        )

                        if rgb_src.nodata is not None:
                            rgb_valid &= (
                                rgb[0]
                                != rgb_src.nodata
                            )

                        # Height mask
                        height_valid = np.isfinite(
                            height[0]
                        )

                        if h_src.nodata is not None:
                            height_valid &= (
                                height[0]
                                != h_src.nodata
                            )

                        # Keep only positive heights
                        height_valid &= (
                            height[0] > 0
                        )

                        if rgb_valid.sum() == 0:
                            continue

                        if height_valid.sum() == 0:
                            continue

                        records.append({
                            "sample":
                                sample,
                            "split":
                                split,
                            "tree_id":
                                tree_id,
                            "area_m2":
                                crown.geometry.area,
                            
                            "Red":
                                rgb[0][rgb_valid].mean(),
                            "Green":
                                rgb[1][rgb_valid].mean(),
                            "Blue":
                                rgb[2][rgb_valid].mean(),
                            "NIR":
                                rgb[3][rgb_valid].mean(),
                            "Height":
                                height[0][height_valid & (height[0] > 0)].mean() #only positive heights
                        })

                    except Exception:
                        print(f"Error in {sample}, tree {tree_id}: {e}")
                        continue

    return pd.DataFrame(records)



def calculate_correlations(df):

    variables = [
        "Red",
        "Green",
        "Blue",
        "NIR",
        "Height",
        "area_m2"
    ]

    pearson = df[variables].corr(method="pearson")
    spearman = df[variables].corr(method="spearman")

    return pearson, spearman

In [ ]:
# setup diplay functions


def show_heatmap(
        corr,
        title):

    plt.figure(
        figsize=(7,6)
    )

    sns.heatmap(
        corr,
        annot=True,
        cmap="RdBu_r",
        center=0,
        vmin=-1,
        vmax=1,
        fmt=".2f",
        square=True
    )

    plt.title(title)

    plt.show()


def show_area_relationships(
        df,
        title):

    features = [
        "Red",
        "Green",
        "Blue",
        "NIR",
        "Height"
    ]

    fig, axes = plt.subplots(
        2,
        3,
        figsize=(14,8)
    )

    axes = axes.flatten()

    for ax, feature in zip(
        axes,
        features
    ):

        sns.regplot(
            data=df,
            x=feature,
            y="area_m2",
            scatter_kws={
                "alpha":0.3,
                "s":10
            },

            line_kws={
                "color":"red"
            },

            ax=ax
        )


        ax.set_title(feature)

    axes[-1].axis("off")


    plt.suptitle(title)
    plt.tight_layout()

    plt.show()

In [ ]:
# variable setup

datasets = {
    "Summer 20cm":(
        image_folders["RGBI_20cm_summer"],
        image_folders["Height_20cm"]
    ),

    "Spring 20cm":(
        image_folders["RGBI_20cm_spring"],
        image_folders["Height_20cm"]
    ),

    "Spring 7.5cm":    (
        image_folders["RGBI_7_5cm"],
        image_folders["Height_7_5cm"]
    )
}


feature_tables = {}
pearson_results = {}
spearman_results = {}

In [ ]:
# --------------------------------------------------------------
# process and display correlations
# --------------------------------------------------------------

for name, (rgb, height) in datasets.items():

    print("\n")
    print("="*70)
    print(name)
    print("="*70)

    df = extract_tree_features(
        rgb,
        height,
        label_folders
    )

    print(
        "Number of crowns:",
        len(df)
    )

    display(
        df.head()
    )
# 
    feature_tables[name] = df


    pearson, spearman = calculate_correlations(df)

    pearson_results[name] = pearson
    spearman_results[name] = spearman


    show_heatmap(
        pearson,
        f"{name} - Pearson"
    )

    show_heatmap(
        spearman,
        f"{name} - Spearman"
    )

    show_area_relationships(
        df,
        f"{name} - Crown area relationships"
    )

**Results:** The positive correlation between RGB is generally, as expected, the strongest. NIR also has a strong positive correlation with RGB, though not as much as RGB itself.
Height correlates negativley with RGBI and positivley with crown area. Negative heights are exluded, as negative heights are occuring in water, not on land. This correlation is also expected, as height is a strong predictor for tree crowns. The correlation is diluted by height also indicating builidngs.

Crown area is positivley correlated to height and NIR, with height being the strongest predicor. For summer images crown area is slightly negativly correlated with RGB, while spring images show a slight positive correlation. NIR is always slightly positivly correlated, but notably stronger in summer than in spring. This is expected, as summer images are leaf-on, therefore showing higher NIR values.

In [ ]:
def compare_area_correlations(results):

    table = []

    for name, corr in results.items():
        values = (
            corr["area_m2"]
            .drop("area_m2")
        )
        values.name = name
        table.append(values)

    return pd.DataFrame(table)



pearson_area = compare_area_correlations(pearson_results)
spearman_area = compare_area_correlations(spearman_results)

In [ ]:
print("Pearson correlations with crown area")
display(pearson_area)


print("Spearman correlations with crown area")
display(spearman_area)

In [ ]:
# spearman correlation with tree crown area

plt.figure(
    figsize=(10,5)
)


sns.heatmap(
    spearman_area,
    annot=True,
    cmap="RdBu_r",
    center=0,
    vmin=-1,
    vmax=1
)


plt.title("Spearman correlation with tree crown area")

plt.xlabel("Feature")
plt.ylabel("Dataset")


plt.show()

**Results:** The comparison between the different images shows clear differences in correlation. Height is, as expected, always positivly correlated to crown area. NIR shows a stronger correlation to crown area in summer than in spring, which is also expected. 

RGB show positve correlations in spring, with blue being the strongest indicator of the three, while showing negative correlations in the summer images. This highlights the challenge of working with images from different seasons and indicated that for high-quality results, seperate models might be needed for spring and summer images.